# DevBugger Data Preparation

Prepare one shared set of training reports and one held-out test set for classification and resolution-time regression. Run this notebook from the project folder in the project Python environment. It uses the report assignments saved in `devbugger_split.csv`.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

reports = pd.read_csv("devbugger_dataset_800.csv").set_index(
    "row_id", verify_integrity=True
)
split_path = Path("devbugger_split.csv")
assert reports.index.notna().all(), "Every report needs a stable row_id."

## Share the same reports across both tasks

The original split used 80% for training, 20% for testing, seed 42, and stratification by `Bug_Type`. The saved IDs preserve each assignment if the CSV is reordered. The creation branch below reproduces the original split from the original dataset order if the split file is missing.

In [ ]:
if split_path.exists():
    assignments = pd.read_csv(split_path).set_index("row_id", verify_integrity=True)
    assert assignments.index.sort_values().equals(reports.index.sort_values()), (
        "The split file and dataset must contain the same report IDs."
    )
    split = assignments["split"].reindex(reports.index)
else:
    train_ids, test_ids = train_test_split(
        reports.index,
        test_size=0.20,
        random_state=42,
        stratify=reports["Bug_Type"],
    )
    split = pd.Series("train", index=reports.index, name="split")
    split.loc[test_ids] = "test"
    split.to_csv(split_path, mode="x")

assert split.isin(["train", "test"]).all(), "Unexpected or missing split label."
train = reports.loc[split.eq("train")]
test = reports.loc[split.eq("test")]
assert len(train) and len(test)
assert train.index.intersection(test.index).empty
print(split.value_counts().to_string())
print(pd.crosstab(reports["Bug_Type"], split)[["train", "test"]])

## Select inputs available when a report is posted

This exercise uses the report title and body, the source platform, and two length measurements. Resolution timestamps and collection-time comments would expose later information. The classification target is `Bug_Type`. The regression target is `resolution_hours`. We will later compare regression with a known bug type against the demo workflow that receives a predicted type.

In [ ]:
numeric_columns = ["title_len_words", "body_len_chars"]
features = reports[["source", *numeric_columns]].copy()
features["report_text"] = (
    reports["report_title"].fillna("") + " " + reports["body"].fillna("")
)

train_inputs = features.loc[train.index]
test_inputs = features.loc[test.index]

## Fit the preprocessor on training reports

TF-IDF learns weights for words and two-word phrases appearing in at least two training reports. One-hot encoding creates one column per observed source. The numeric pipeline fills any missing length with its training median and standardizes each length using training statistics.

In [ ]:
preprocessor = ColumnTransformer(
    [
        ("text", TfidfVectorizer(ngram_range=(1, 2), min_df=2), "report_text"),
        ("source", OneHotEncoder(handle_unknown="ignore"), ["source"]),
        (
            "numeric",
            make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
            numeric_columns,
        ),
    ]
)

train_features = preprocessor.fit_transform(train_inputs)
test_features = preprocessor.transform(test_inputs)

print("Training matrix:", train_features.shape)
print("Test matrix:", test_features.shape)
for name, columns in preprocessor.output_indices_.items():
    if columns.stop > columns.start:
        print(f"{name}: {columns.stop - columns.start:,} features")

The training and test matrices use the same columns because the test reports pass through the fitted preprocessor. During model selection, put the preprocessor and estimator in one scikit-learn `Pipeline` and pass the original training inputs into cross-validation. Each fold then learns its own vocabulary, category encoding, and numeric statistics.